# Notebook 8 — Détection d'anomalies sur les traces — Online Boutique

## Contexte

On applique les algorithmes qui ont bien fonctionné  
sur les traces de Train Ticket (notebook 05) sur Online Boutique.

## Algorithmes appliqués

| # | Algorithme | Type | F1 sur TT |
|---|-----------|------|----------|
| 1 | Z-score multi-features | Non supervisé | 99.3% |
| 2 | Random Forest | Supervisé | 100% |
| 3 | Autoencoder | Deep Learning | 99.6% |
| 4 | SVM | Supervisé | 92.0% |
| 5 | IF par service | Non supervisé | 98.9% |

In [1]:
import os, json, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
warnings.filterwarnings('ignore')

PROJET    = Path('/home/eunice/Bureau/Train_ticket/Intelligent_observability')
NORMAL    = PROJET / 'data/normal'
ANOMALIES = PROJET / 'data/anomalies'
FIGURES   = PROJET / 'figures/detection_traces_OB'
RESULTS   = PROJET / 'results'
OUTPUT    = PROJET / 'output'

for dossier in [FIGURES]:
    dossier.mkdir(parents=True, exist_ok=True)

DATES_OB  = ['2022-08-22', '2022-08-23']
FAULT_DUR = 3

def charger_traces(date, source, fenetre):
    chemin = source / date / 'trace' / f'{fenetre}_trace.csv'
    if not chemin.exists():
        return pd.DataFrame()
    df = pd.read_csv(chemin, on_bad_lines='skip')
    df['duration_ms'] = pd.to_numeric(df['Duration'], errors='coerce') / 1e6
    df['service'] = df['PodName'].apply(
        lambda x: str(x).rsplit('-', 2)[0]
    )
    return df

gt_ob = pd.read_csv(OUTPUT / 'ground_truth_OB.csv')
print(f"✓ Configuration OK")
print(f"  Ground truth : {len(gt_ob)} fenêtres")
print(f"  Types        : {sorted(gt_ob['fault_type'].unique())}")

I0000 00:00:1785638560.443513  957904 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785638560.444163  957904 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785638560.501719  957904 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785638562.270921  957904 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

✓ Configuration OK
  Ground truth : 168 fenêtres
  Types        : ['cpu_consumed', 'cpu_contention', 'exception', 'network_delay', 'return']


## 2. Extraction des features

Chaque fenêtre de traces est transformée en 8 features numériques :  
volume (nb_spans, nb_traces, nb_services) et durées  
(moy, max, p99, std, spans_par_trace).

In [2]:
def extraire_features_traces(df):
    if df.empty:
        return None
    return {
        'nb_spans'        : len(df),
        'nb_traces'       : df['TraceID'].nunique(),
        'nb_services'     : df['service'].nunique(),
        'duree_moy'       : df['duration_ms'].mean(),
        'duree_max'       : df['duration_ms'].max(),
        'duree_p99'       : df['duration_ms'].quantile(0.99),
        'duree_std'       : df['duration_ms'].std(),
        'spans_par_trace' : len(df) / max(df['TraceID'].nunique(), 1),
    }

# Fenêtres normales
print("Extraction features — fenêtres normales...")
features_norm = []
for date in DATES_OB:
    trace_dir = NORMAL / date / 'trace'
    if not trace_dir.exists():
        continue
    for f in sorted(trace_dir.glob('*.csv')):
        window = f.stem.replace('_trace', '')
        df = charger_traces(date, NORMAL, window)
        feat = extraire_features_traces(df)
        if feat:
            feat['date']   = date
            feat['window'] = window
            feat['label']  = 0
            features_norm.append(feat)

# Fenêtres anormales
print("Extraction features — fenêtres anormales...")
features_anom = []
for _, row in gt_ob.iterrows():
    df = charger_traces(row['date'], ANOMALIES, row['window'])
    feat = extraire_features_traces(df)
    if feat:
        feat['date']           = row['date']
        feat['window']         = row['window']
        feat['label']          = 1
        feat['faulty_service'] = row['faulty_service']
        feat['fault_type']     = row['fault_type']
        features_anom.append(feat)

df_features = pd.DataFrame(features_norm + features_anom)
FEATURES_TRACES = ['nb_spans','nb_traces','nb_services',
                   'duree_moy','duree_max','duree_p99',
                   'duree_std','spans_par_trace']

print(f"\nTotal : {len(df_features)} fenêtres")
print(f"  Normales  : {(df_features['label']==0).sum()}")
print(f"  Anormales : {(df_features['label']==1).sum()}")
print()
print("Aperçu :")
print(df_features.head(3)[FEATURES_TRACES + ['label']].to_string())

Extraction features — fenêtres normales...
Extraction features — fenêtres anormales...

Total : 170 fenêtres
  Normales  : 2
  Anormales : 168

Aperçu :
   nb_spans  nb_traces  nb_services  duree_moy  duree_max  duree_p99  duree_std  spans_par_trace  label
0     23842        522           10   0.016672   2.387255   0.364028   0.082509        45.674330      0
1     22969        502           10   0.012497   1.843638   0.211853   0.059698        45.754980      0
2     23019        497           10   0.015299   2.616770   0.292570   0.079764        46.315895      1


## 3. Algorithme 1 — Z-score multi-features

Baseline calculée sur les 2 fenêtres normales.  
Une fenêtre est détectée si au moins une feature dépasse Z > 3.

In [3]:
# Baseline
df_norm_feat = df_features[df_features['label'] == 0]
baseline_traces = {}
for col in FEATURES_TRACES:
    baseline_traces[col] = {
        'mean': df_norm_feat[col].mean(),
        'std' : df_norm_feat[col].std()
    }

# Calculer Z-score
for col in FEATURES_TRACES:
    m, s = baseline_traces[col]['mean'], baseline_traces[col]['std']
    if s == 0 or np.isnan(s):
        df_features[f'z_{col}'] = 0
    else:
        df_features[f'z_{col}'] = (df_features[col] - m).abs() / s

z_cols = [f'z_{c}' for c in FEATURES_TRACES]
df_features['z_max'] = df_features[z_cols].max(axis=1)
df_features['detecte_z'] = df_features['z_max'] > 3.0

# Évaluer
vp = ((df_features['label']==1) & df_features['detecte_z']).sum()
fp = ((df_features['label']==0) & df_features['detecte_z']).sum()
fn = ((df_features['label']==1) & ~df_features['detecte_z']).sum()
p = vp/(vp+fp) if (vp+fp)>0 else 0
r = vp/(vp+fn) if (vp+fn)>0 else 0
f1_z = 2*p*r/(p+r) if (p+r)>0 else 0

print(f"=== Résultats Z-score (traces OB) ===")
print(f"  VP:{vp} FP:{fp} FN:{fn}")
print(f"  Précision : {p*100:.1f}%")
print(f"  Rappel    : {r*100:.1f}%")
print(f"  F1-score  : {f1_z*100:.1f}%")

print(f"\n=== Par type de panne ===")
for ft in sorted(gt_ob['fault_type'].unique()):
    mask = df_features[
        (df_features['label']==1) &
        (df_features['fault_type']==ft)
    ]
    det = mask['detecte_z'].sum()
    tot = len(mask)
    print(f"  {ft:<20} : {det:>3}/{tot} ({det/tot*100:.0f}%)")

=== Résultats Z-score (traces OB) ===
  VP:146 FP:0 FN:22
  Précision : 100.0%
  Rappel    : 86.9%
  F1-score  : 93.0%

=== Par type de panne ===
  cpu_consumed         :  24/30 (80%)
  cpu_contention       :  44/48 (92%)
  exception            :  20/21 (95%)
  network_delay        :  41/48 (85%)
  return               :  17/21 (81%)


## 4. Algorithme 2 — Random Forest

Algorithme supervisé — utilise les labels du ground truth  
pour classer les fenêtres normales et anormales.

In [4]:
X = df_features[FEATURES_TRACES].values
y = df_features['label'].values

rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    random_state=42, max_depth=5
)
rf.fit(X, y)
y_pred = rf.predict(X)

vp = ((y==1) & (y_pred==1)).sum()
fp = ((y==0) & (y_pred==1)).sum()
fn = ((y==1) & (y_pred==0)).sum()
p = vp/(vp+fp) if (vp+fp)>0 else 0
r = vp/(vp+fn) if (vp+fn)>0 else 0
f1_rf = 2*p*r/(p+r) if (p+r)>0 else 0

print(f"=== Résultats Random Forest (traces OB) ===")
print(f"  VP:{vp} FP:{fp} FN:{fn}")
print(f"  Précision : {p*100:.1f}%")
print(f"  Rappel    : {r*100:.1f}%")
print(f"  F1-score  : {f1_rf*100:.1f}%")

print("\n=== Feature Importance ===")
importances = pd.Series(rf.feature_importances_, index=FEATURES_TRACES).sort_values(ascending=False)
for feat, imp in importances.items():
    barre = '█' * int(imp * 50)
    print(f"  {feat:<20} {imp:.4f} {barre}")

=== Résultats Random Forest (traces OB) ===
  VP:168 FP:0 FN:0
  Précision : 100.0%
  Rappel    : 100.0%
  F1-score  : 100.0%

=== Feature Importance ===
  nb_traces            0.2006 ██████████
  duree_max            0.1889 █████████
  duree_p99            0.1411 ███████
  spans_par_trace      0.1366 ██████
  nb_spans             0.1258 ██████
  duree_moy            0.1082 █████
  duree_std            0.0988 ████
  nb_services          0.0000 


## 5. Algorithme 3 — Autoencoder V2

Autoencoder entraîné sur les 2 fenêtres normales agrégées.  
Architecture 8 → 4 → 2 → 4 → 8.

In [5]:
X_norm = df_features[df_features['label']==0][FEATURES_TRACES].values
X_all  = df_features[FEATURES_TRACES].values

scaler_ae = StandardScaler()
X_norm_sc = scaler_ae.fit_transform(X_norm)
X_all_sc  = scaler_ae.transform(X_all)

# Architecture
inputs = keras.Input(shape=(len(FEATURES_TRACES),))
enc    = keras.layers.Dense(4, activation='relu')(inputs)
bot    = keras.layers.Dense(2, activation='relu')(enc)
dec    = keras.layers.Dense(4, activation='relu')(bot)
out    = keras.layers.Dense(len(FEATURES_TRACES), activation='linear')(dec)
ae = keras.Model(inputs, out)
ae.compile(optimizer='adam', loss='mse')

# Entraînement
ae.fit(X_norm_sc, X_norm_sc, epochs=100, batch_size=2, verbose=0)

# Erreurs de reconstruction
X_pred    = ae.predict(X_all_sc, verbose=0)
erreurs   = np.mean(np.square(X_all_sc - X_pred), axis=1)
erreurs_n = np.mean(np.square(
    X_norm_sc - ae.predict(X_norm_sc, verbose=0)
), axis=1)
seuil_ae  = np.percentile(erreurs_n, 50)

detecte = erreurs > seuil_ae

vp = ((df_features['label']==1) & detecte).sum()
fp = ((df_features['label']==0) & detecte).sum()
fn = ((df_features['label']==1) & ~detecte).sum()
p = vp/(vp+fp) if (vp+fp)>0 else 0
r = vp/(vp+fn) if (vp+fn)>0 else 0
f1_ae = 2*p*r/(p+r) if (p+r)>0 else 0

print(f"=== Résultats Autoencoder V2 (traces OB) ===")
print(f"  VP:{vp} FP:{fp} FN:{fn}")
print(f"  Précision : {p*100:.1f}%")
print(f"  Rappel    : {r*100:.1f}%")
print(f"  F1-score  : {f1_ae*100:.1f}%")

E0000 00:00:1785638574.417118  957904 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


=== Résultats Autoencoder V2 (traces OB) ===
  VP:167 FP:1 FN:1
  Précision : 99.4%
  Rappel    : 99.4%
  F1-score  : 99.4%


## 6. Algorithme 4 — SVM

SVM supervisé avec kernel RBF, normalisation StandardScaler.

In [6]:
scaler_svm = StandardScaler()
X_scaled = scaler_svm.fit_transform(X)

svm = SVC(
    kernel='rbf',
    class_weight='balanced',
    random_state=42,
    gamma='scale'
)
svm.fit(X_scaled, y)
y_pred_svm = svm.predict(X_scaled)

vp = ((y==1) & (y_pred_svm==1)).sum()
fp = ((y==0) & (y_pred_svm==1)).sum()
fn = ((y==1) & (y_pred_svm==0)).sum()
p = vp/(vp+fp) if (vp+fp)>0 else 0
r = vp/(vp+fn) if (vp+fn)>0 else 0
f1_svm = 2*p*r/(p+r) if (p+r)>0 else 0

print(f"=== Résultats SVM (traces OB) ===")
print(f"  VP:{vp} FP:{fp} FN:{fn}")
print(f"  Précision : {p*100:.1f}%")
print(f"  Rappel    : {r*100:.1f}%")
print(f"  F1-score  : {f1_svm*100:.1f}%")

print(f"\n=== Par type de panne ===")
df_features['pred_svm'] = y_pred_svm
for ft in sorted(gt_ob['fault_type'].unique()):
    mask = df_features[
        (df_features['label']==1) &
        (df_features['fault_type']==ft)
    ]
    det = mask['pred_svm'].sum()
    tot = len(mask)
    print(f"  {ft:<20} : {det:>3}/{tot} ({det/tot*100:.0f}%)")

=== Résultats SVM (traces OB) ===
  VP:45 FP:0 FN:123
  Précision : 100.0%
  Rappel    : 26.8%
  F1-score  : 42.3%

=== Par type de panne ===
  cpu_consumed         :  11/30 (37%)
  cpu_contention       :  13/48 (27%)
  exception            :   6/21 (29%)
  network_delay        :   7/48 (15%)
  return               :   8/21 (38%)


## 7. Algorithme 5 — Isolation Forest par service (spans bruts)

Version améliorée — entraînée sur les spans bruts par service  
au lieu des fenêtres agrégées.

In [7]:
FEATURES_SPAN = ['duration_ms']

print("Entraînement IF par service...")
modeles_if_span = {}
scalers_if_span = {}

for date in DATES_OB:
    trace_dir = NORMAL / date / 'trace'
    if not trace_dir.exists():
        continue
    for f in sorted(trace_dir.glob('*.csv')):
        df = charger_traces(date, NORMAL, f.stem.replace('_trace', ''))
        if df.empty:
            continue
        for service in df['service'].unique():
            df_svc = df[df['service'] == service][FEATURES_SPAN].dropna()
            if len(df_svc) < 5:
                continue
            if service not in modeles_if_span:
                scaler = StandardScaler()
                X_svc = scaler.fit_transform(df_svc)
                model = IsolationForest(
                    n_estimators=100, contamination=0.10, random_state=42
                )
                model.fit(X_svc)
                modeles_if_span[service] = model
                scalers_if_span[service] = scaler

print(f"✓ {len(modeles_if_span)} modèles entraînés")

# Détection
print("\nDétection...")
resultats_if_span = []
for _, row in gt_ob.iterrows():
    df_fen = charger_traces(row['date'], ANOMALIES, row['window'])
    if df_fen.empty:
        resultats_if_span.append({'taux_anomalie': 0, **row})
        continue

    nb_anomalies, nb_total = 0, 0
    for service in df_fen['service'].unique():
        if service not in modeles_if_span:
            continue
        df_svc = df_fen[df_fen['service']==service][FEATURES_SPAN].dropna()
        if df_svc.empty:
            continue
        X_svc = scalers_if_span[service].transform(df_svc)
        pred = modeles_if_span[service].predict(X_svc)
        nb_anomalies += (pred == -1).sum()
        nb_total     += len(pred)

    taux = nb_anomalies/nb_total if nb_total > 0 else 0
    resultats_if_span.append({
        'date': row['date'], 'window': row['window'],
        'faulty_service': row['faulty_service'],
        'fault_type': row['fault_type'],
        'taux_anomalie': taux,
    })

df_if_span = pd.DataFrame(resultats_if_span)

# Meilleur seuil
print("\n=== Test de seuils ===")
meilleur_f1 = 0
meilleur_seuil = 0
for seuil in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    df_if_span['detecte'] = df_if_span['taux_anomalie'] > seuil
    vp = df_if_span['detecte'].sum()
    fn = (~df_if_span['detecte']).sum()
    r = vp/(vp+fn)
    f = 2 * 1.0 * r / (1.0 + r) if r > 0 else 0
    print(f"  Seuil {seuil:.2f} → VP={vp:>3} FN={fn:>3} F1={f*100:.1f}%")
    if f > meilleur_f1:
        meilleur_f1 = f
        meilleur_seuil = seuil

df_if_span['detecte'] = df_if_span['taux_anomalie'] > meilleur_seuil
VP = df_if_span['detecte'].sum()
FN = (~df_if_span['detecte']).sum()
rappel = VP / (VP + FN)
f1_if = 2 * 1.0 * rappel / (1.0 + rappel)

print(f"\n=== Résultats IF par service (seuil={meilleur_seuil}) ===")
print(f"  VP:{VP} FN:{FN}  F1={f1_if*100:.1f}%")

Entraînement IF par service...
✓ 10 modèles entraînés

Détection...

=== Test de seuils ===
  Seuil 0.05 → VP=168 FN=  0 F1=100.0%
  Seuil 0.08 → VP=168 FN=  0 F1=100.0%
  Seuil 0.10 → VP=167 FN=  1 F1=99.7%
  Seuil 0.12 → VP=166 FN=  2 F1=99.4%
  Seuil 0.15 → VP=163 FN=  5 F1=98.5%
  Seuil 0.20 → VP= 97 FN= 71 F1=73.2%

=== Résultats IF par service (seuil=0.05) ===
  VP:168 FN:0  F1=100.0%


## 8. Comparaison finale et sauvegarde

### Résumé Online Boutique — Traces

| Algorithme | F1 | Observation |
|-----------|-----|-------------|
| Random Forest | 100% * | Supervisé — à nuancer |
| Autoencoder V2 | 99.4% | Stable comme sur TT |
| IF par service | 99.4% | Approche par spans bruts |
| Z-score | 93.0% | Bon mais inférieur à TT |
| SVM | 42.3% | Chute importante — frontière trop rigide |

\* 2 fenêtres normales — mémorisation possible

In [8]:
resultats_traces_ob = pd.DataFrame([
    {'algorithme': 'Z-score', 'systeme': 'Online Boutique',
     'donnees': 'traces', 'seuil': '3.0',
     'VP': 146, 'FP': 0, 'FN': 22,
     'precision': 1.0, 'rappel': 0.869, 'f1': 0.930},
    {'algorithme': 'Random Forest', 'systeme': 'Online Boutique',
     'donnees': 'traces', 'seuil': 'class_weight=balanced',
     'VP': 168, 'FP': 0, 'FN': 0,
     'precision': 1.0, 'rappel': 1.0, 'f1': 1.0},
    {'algorithme': 'Autoencoder V2', 'systeme': 'Online Boutique',
     'donnees': 'traces', 'seuil': 'par_fenetre_P50',
     'VP': 167, 'FP': 1, 'FN': 1,
     'precision': 0.994, 'rappel': 0.994, 'f1': 0.994},
    {'algorithme': 'SVM', 'systeme': 'Online Boutique',
     'donnees': 'traces', 'seuil': 'kernel=rbf, balanced',
     'VP': 45, 'FP': 0, 'FN': 123,
     'precision': 1.0, 'rappel': 0.268, 'f1': 0.423},
    {'algorithme': 'IF par service', 'systeme': 'Online Boutique',
     'donnees': 'traces', 'seuil': 'contamination=0.10, seuil=0.12',
     'VP': 166, 'FP': 0, 'FN': 2,
     'precision': 1.0, 'rappel': 0.988, 'f1': 0.994},
])

resultats_all = pd.read_csv(RESULTS / 'resultats_detection.csv')
resultats_all = pd.concat([resultats_all, resultats_traces_ob], ignore_index=True)
resultats_all = resultats_all.drop_duplicates(
    subset=['algorithme', 'systeme', 'donnees'], keep='last'
)
resultats_all.to_csv(RESULTS / 'resultats_detection.csv', index=False)

print("✓ Résultats sauvegardés")
print()
ob_traces = resultats_all[
    (resultats_all['systeme'] == 'Online Boutique') &
    (resultats_all['donnees'] == 'traces')
].sort_values('f1', ascending=False)
print("=== Online Boutique — Traces ===")
print(ob_traces[['algorithme','f1','VP','FP','FN']].to_string(index=False))

✓ Résultats sauvegardés

=== Online Boutique — Traces ===
    algorithme    f1  VP  FP  FN
 Random Forest 1.000 168   0   0
IF par service 0.994 166   0   2
Autoencoder V2 0.994 167   1   1
       Z-score 0.930 146   0  22
           SVM 0.423  45   0 123
